In [ ]:
!unzip RGB.zip

Archive:  RGB.zip
   creating: RGB/
   creating: RGB/Patches/
   creating: RGB/Patches/Abnormal(Ulcer)/
  inflating: RGB/Patches/Abnormal(Ulcer)/1.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/10.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/100.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/101.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/102.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/103.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/104.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/105.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/106.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/107.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/108.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/109.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/11.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/110.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/111.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/112.jpg  
  inflating: RGB/Patches/Abnormal(Ulcer)/113.jpg  
  inflating: RGB/Patches/Abnormal

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
RGB_PATH = "/content/RGB"

In [ ]:
# Separate count images
def separate_and_count():

    rgb = Path(RGB_PATH)

    total_images = 0
    structure = {}

    # Patches folder
    patches = rgb / "Patches"
    if patches.exists():
        print("\nPATCHES FOLDER:")
        structure['Patches'] = {}
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                count = len(images)
                structure['Patches'][subfolder] = count
                print(f"  {subfolder}: {count} images")
                total_images += count
            else:
                print(f"  {subfolder}: Not found")

    # TestSet folder
    testset = rgb / "TestSet"
    if testset.exists():
        images = [f for f in testset.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
        count = len(images)
        structure['TestSet'] = count
        print(f"\nTESTSET FOLDER: {count} images")
        total_images += count

    # Transfer-Learning images folder
    transfer = rgb / "Transfer-Learning images"
    if transfer.exists():
        print("\nTRANSFER-LEARNING IMAGES:")
        structure['Transfer-Learning'] = {}
        for subfolder in ['samples', 'internetSet', 'Wound Images', 'Wound Images2']:
            folder = transfer / subfolder
            if folder.exists():
                # Count all images recursively if needed
                images = list(folder.rglob('*'))
                images = [f for f in images if f.is_file() and f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                count = len(images)
                structure['Transfer-Learning'][subfolder] = count
                print(f"  {subfolder}: {count} images")
                total_images += count
            else:
                print(f"  {subfolder}: Not found")

    print(f"\nTotal images: {total_images}")

    return structure

separate_and_count()


PATCHES FOLDER:
  Abnormal(Ulcer): 512 images
  Normal(Healthy skin): 543 images

TESTSET FOLDER: 163 images

TRANSFER-LEARNING IMAGES:
  samples: 36 images
  internetSet: 134 images
  Wound Images: 109 images
  Wound Images2: 672 images

Total images: 2169


{'Patches': {'Abnormal(Ulcer)': 512, 'Normal(Healthy skin)': 543},
 'TestSet': 163,
 'Transfer-Learning': {'samples': 36,
  'internetSet': 134,
  'Wound Images': 109,
  'Wound Images2': 672}}

In [ ]:
# check blurry images
def check_blur():

    rgb = Path(RGB_PATH)

    # Collect all images
    all_images = []

    patches = rgb / "Patches"
    if patches.exists():
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                all_images.extend(images)

    testset = rgb / "TestSet"
    if testset.exists():
        images = [f for f in testset.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
        all_images.extend(images)

    print(f"\nChecking {len(all_images)} images for blur...")

    BLUR_THRESHOLD = 100
    blurry_images = []

    for img_path in all_images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

        if laplacian_var < BLUR_THRESHOLD:
            blurry_images.append((img_path.name, laplacian_var))

    print(f"\nBlurry images found: {len(blurry_images)} ({len(blurry_images)/len(all_images)*100:.1f}%)")

    if len(blurry_images) > 0 and len(blurry_images) < 20:
        print("\nBlurry image list:")
        for name, score in blurry_images[:10]:
            print(f"  {name}: {score:.2f}")

    return blurry_images

check_blur()


Checking 1218 images for blur...

Blurry images found: 450 (36.9%)


[('49.jpg', np.float64(27.692341941935172)),
 ('271.jpg', np.float64(63.41382020198022)),
 ('48.jpg', np.float64(8.7304786382641)),
 ('189.jpg', np.float64(76.25585385711429)),
 ('310.jpg', np.float64(41.98260043929488)),
 ('137.jpg', np.float64(9.283249558467194)),
 ('450.jpg', np.float64(16.906044396794076)),
 ('330.jpg', np.float64(40.560468317451175)),
 ('380.jpg', np.float64(23.720738521768176)),
 ('480.jpg', np.float64(72.06542749496263)),
 ('485.jpg', np.float64(55.097066962287016)),
 ('427.jpg', np.float64(80.31216007687061)),
 ('260.jpg', np.float64(18.411160768940267)),
 ('438.jpg', np.float64(19.678136454180645)),
 ('428.jpg', np.float64(32.37870250131527)),
 ('273.jpg', np.float64(73.99649110132333)),
 ('3.jpg', np.float64(53.21889448503513)),
 ('359.jpg', np.float64(62.72482547497858)),
 ('50.jpg', np.float64(61.681447544677816)),
 ('345.jpg', np.float64(16.247193933476613)),
 ('373.jpg', np.float64(33.526130453441)),
 ('54.jpg', np.float64(54.727647696371534)),
 ('371.jpg

In [ ]:
# check visibility

def check_visibility():

    rgb = Path(RGB_PATH)

    # Collect all images
    all_images = []

    patches = rgb / "Patches"
    if patches.exists():
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                all_images.extend(images)

    testset = rgb / "TestSet"
    if testset.exists():
        images = [f for f in testset.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
        all_images.extend(images)

    print(f"\nChecking {len(all_images)} images for visibility...")

    VISIBILITY_THRESHOLD = 0.5  # % of edge pixels
    low_visibility = []

    for img_path in all_images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)

        total_pixels = edges.shape[0] * edges.shape[1]
        edge_pixels = np.sum(edges > 0)
        visibility = (edge_pixels / total_pixels) * 100

        if visibility < VISIBILITY_THRESHOLD:
            low_visibility.append((img_path.name, visibility))

    print(f"\nLow visibility images: {len(low_visibility)} ({len(low_visibility)/len(all_images)*100:.1f}%)")

    if len(low_visibility) > 0 and len(low_visibility) < 20:
        print("\nLow visibility list:")
        for name, vis in low_visibility[:10]:
            print(f"  {name}: {vis:.2f}%")

    return low_visibility

check_visibility()


Checking 1218 images for visibility...

Low visibility images: 153 (12.6%)


[('48.jpg', np.float64(0.05779655612244898)),
 ('450.jpg', np.float64(0.22122130102040816)),
 ('9.jpg', np.float64(0.3706951530612245)),
 ('30.jpg', np.float64(0.05779655612244898)),
 ('62.jpg', np.float64(0.01594387755102041)),
 ('338.jpg', np.float64(0.0)),
 ('5.jpg', np.float64(0.08769132653061225)),
 ('342.jpg', np.float64(0.20129145408163263)),
 ('504.jpg', np.float64(0.12356505102040817)),
 ('173.jpg', np.float64(0.09367028061224489)),
 ('58.jpg', np.float64(0.0)),
 ('310.jpg', np.float64(0.42849170918367346)),
 ('268.jpg', np.float64(0.013950892857142856)),
 ('137.jpg', np.float64(0.2969547193877551)),
 ('6.jpg', np.float64(0.45639349489795916)),
 ('450.jpg', np.float64(0.0)),
 ('380.jpg', np.float64(0.49824617346938777)),
 ('480.jpg', np.float64(0.0)),
 ('485.jpg', np.float64(0.1753826530612245)),
 ('392.jpg', np.float64(0.01992984693877551)),
 ('156.jpg', np.float64(0.4245057397959184)),
 ('428.jpg', np.float64(0.0896843112244898)),
 ('359.jpg', np.float64(0.2590880102040816))

In [ ]:
# verify color space
def verify_color_space():

    rgb = Path(RGB_PATH)

    # Collect sample images
    all_images = []

    patches = rgb / "Patches"
    if patches.exists():
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                all_images.extend(images[:10])

    print(f"\nChecking color space on {len(all_images)} sample images...")

    color_modes = {}

    for img_path in all_images:
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue

        if len(img.shape) == 2:
            mode = "Grayscale"
        elif len(img.shape) == 3 and img.shape[2] == 3:
            mode = "3 channels (RGB)"
        elif len(img.shape) == 3 and img.shape[2] == 4:
            mode = "4 channels (RGBA)"
        else:
            mode = f"Unknown"

        color_modes[mode] = color_modes.get(mode, 0) + 1

    print("\nColor modes:")
    for mode, count in color_modes.items():
        print(f"  {mode}: {count} images")

verify_color_space()


Checking color space on 20 sample images...

Color modes:
  3 channels (RGB): 20 images


In [ ]:
# check resolution

def check_resolutions():

    rgb = Path(RGB_PATH)

    # Collect all images
    all_images = []

    patches = rgb / "Patches"
    if patches.exists():
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                all_images.extend(images)

    testset = rgb / "TestSet"
    if testset.exists():
        images = [f for f in testset.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
        all_images.extend(images)

    transfer = rgb / "Transfer-Learning images"
    if transfer.exists():
        for subfolder in ['samples', 'internetSet', 'Wound Images', 'Wound Images2']:
            folder = transfer / subfolder
            if folder.exists():
                images = list(folder.rglob('*'))
                images = [f for f in images if f.is_file() and f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]
                all_images.extend(images)

    print(f"\nChecking {len(all_images)} images...")

    resolutions = {}

    for img_path in all_images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        h, w = img.shape[:2]
        res = f"{w}x{h}"
        resolutions[res] = resolutions.get(res, 0) + 1

    print("\nResolutions found:")
    sorted_res = sorted(resolutions.items(), key=lambda x: x[1], reverse=True)
    for res, count in sorted_res[:10]:
        print(f"  {res}: {count} images")

    if len(resolutions) > 10:
        print(f"  ... and {len(resolutions)-10} more")

check_resolutions()


Checking 2169 images...

Resolutions found:
  224x224: 1101 images
  259x194: 82 images
  275x183: 72 images
  265x190: 38 images
  273x185: 28 images
  194x259: 27 images
  225x225: 26 images
  276x183: 25 images
  300x168: 23 images
  272x185: 16 images
  ... and 315 more


In [ ]:
"""
RGB DATA CLEANING - FIX ISSUES
Based on analysis:
1. Remove 51 blurry images
2. Remove 153 low visibility images
3. Resize all to 224x224
"""

import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import shutil


RGB_PATH = "/content/RGB"
OUTPUT_PATH = "/content/RGB_C"

BLUR_THRESHOLD = 100
VISIBILITY_THRESHOLD = 0.5  # % edge pixels


def check_blur(img):
    """Check if image is blurry"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var, laplacian_var < BLUR_THRESHOLD

def check_visibility(img):
    """Check if image has sufficient visibility"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    total_pixels = edges.shape[0] * edges.shape[1]
    edge_pixels = np.sum(edges > 0)
    visibility = (edge_pixels / total_pixels) * 100
    return visibility, visibility < VISIBILITY_THRESHOLD


def clean_rgb_data():

    rgb = Path(RGB_PATH)
    output = Path(OUTPUT_PATH)

    # Create output directory structure
    print("Creating output directory structure...")

    # Patches
    (output / "Patches" / "Abnormal(Ulcer)").mkdir(parents=True, exist_ok=True)
    (output / "Patches" / "Normal(Healthy skin)").mkdir(parents=True, exist_ok=True)

    # TestSet
    (output / "TestSet").mkdir(parents=True, exist_ok=True)

    # Transfer-Learning
    for subfolder in ['samples', 'internetSet', 'Wound Images', 'Wound Images2']:
        (output / "Transfer-Learning images" / subfolder).mkdir(parents=True, exist_ok=True)

    print(f"Output directory: {output}\n")

    # Statistics
    stats = {
        'total': 0,
        'removed_blur': 0,
        'removed_visibility': 0,
        'resized': 0,
        'kept': 0
    }

    # Process Patches
    print("Processing PATCHES...")
    for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
        input_folder = rgb / "Patches" / subfolder
        output_folder = output / "Patches" / subfolder

        if not input_folder.exists():
            continue

        images = [f for f in input_folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]

        for img_path in tqdm(images, desc=f"  {subfolder}"):
            stats['total'] += 1

            # Read image
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # Check blur
            blur_score, is_blurry = check_blur(img)
            if is_blurry:
                stats['removed_blur'] += 1
                continue

            # Check visibility
            visibility, low_vis = check_visibility(img)
            if low_vis:
                stats['removed_visibility'] += 1
                continue

            # Resize if needed
            if img.shape[:2] != (224, 224):
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
                stats['resized'] += 1

            # Save
            output_file = output_folder / img_path.name
            cv2.imwrite(str(output_file), img)
            stats['kept'] += 1

    # Process TestSet
    print("\nProcessing TESTSET...")
    input_folder = rgb / "TestSet"
    output_folder = output / "TestSet"

    if input_folder.exists():
        images = [f for f in input_folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]

        for img_path in tqdm(images, desc="  TestSet"):
            stats['total'] += 1

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            blur_score, is_blurry = check_blur(img)
            if is_blurry:
                stats['removed_blur'] += 1
                continue

            visibility, low_vis = check_visibility(img)
            if low_vis:
                stats['removed_visibility'] += 1
                continue

            if img.shape[:2] != (224, 224):
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
                stats['resized'] += 1

            output_file = output_folder / img_path.name
            cv2.imwrite(str(output_file), img)
            stats['kept'] += 1

    # Process Transfer-Learning
    print("\nProcessing TRANSFER-LEARNING...")
    for subfolder in ['samples', 'internetSet', 'Wound Images', 'Wound Images2']:
        input_folder = rgb / "Transfer-Learning images" / subfolder
        output_folder = output / "Transfer-Learning images" / subfolder

        if not input_folder.exists():
            continue

        images = list(input_folder.rglob('*'))
        images = [f for f in images if f.is_file() and f.suffix.lower() in ['.jpg', '.png', '.jpeg', '.bmp']]

        for img_path in tqdm(images, desc=f"  {subfolder}"):
            stats['total'] += 1

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            blur_score, is_blurry = check_blur(img)
            if is_blurry:
                stats['removed_blur'] += 1
                continue

            visibility, low_vis = check_visibility(img)
            if low_vis:
                stats['removed_visibility'] += 1
                continue

            if img.shape[:2] != (224, 224):
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
                stats['resized'] += 1

            output_file = output_folder / img_path.name
            cv2.imwrite(str(output_file), img)
            stats['kept'] += 1

    # Print statistics
    print("\n" + "=" * 60)
    print("CLEANING COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total images processed: {stats['total']}")
    print(f"  Removed (blurry): {stats['removed_blur']}")
    print(f"  Removed (low visibility): {stats['removed_visibility']}")
    print(f"  Resized to 224x224: {stats['resized']}")
    print(f"  Final dataset size: {stats['kept']} images")
    print(f"\n  Removal rate: {((stats['removed_blur'] + stats['removed_visibility']) / stats['total'] * 100):.1f}%")
    print(f"\nCleaned data saved to: {output}")


def verify_cleaned_data():
    """Verify cleaned dataset"""

    print("\n" + "=" * 60)
    print("VERIFYING CLEANED DATA")
    print("=" * 60)

    output = Path(OUTPUT_PATH)

    resolutions = {}
    total = 0

    # Check Patches
    patches = output / "Patches"
    if patches.exists():
        for subfolder in ['Abnormal(Ulcer)', 'Normal(Healthy skin)']:
            folder = patches / subfolder
            if folder.exists():
                images = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
                for img_path in images:
                    img = cv2.imread(str(img_path))
                    if img is not None:
                        h, w = img.shape[:2]
                        res = f"{w}x{h}"
                        resolutions[res] = resolutions.get(res, 0) + 1
                        total += 1

    # Check TestSet
    testset = output / "TestSet"
    if testset.exists():
        images = list(testset.glob('*.jpg')) + list(testset.glob('*.png'))
        for img_path in images:
            img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]
                res = f"{w}x{h}"
                resolutions[res] = resolutions.get(res, 0) + 1
                total += 1

    print(f"\nTotal images in cleaned dataset: {total}")

    print(f"\nResolutions:")
    for res, count in resolutions.items():
        status = "y" if res == "224x224" else "n"
        print(f"  {status} {res}: {count} images")

    if len(resolutions) == 1 and "224x224" in resolutions:
        print("\nALL IMAGES ARE 224x224!")
    else:
        print("\nSome images still have wrong resolution")


if __name__ == "__main__":
    print("=" * 60)
    print("RGB DATA CLEANING - FIX ISSUES")
    print("=" * 60)
    print(f"\nInput: {RGB_PATH}")
    print(f"Output: {OUTPUT_PATH}")
    print(f"\nWill remove:")
    print(f"  - Blurry images (score < 100)")
    print(f"  - Low visibility images (< 0.5% edges)")
    print(f"  - Resize all to 224x224")

    response = input("\nStart cleaning? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Clean
    clean_rgb_data()

    # Verify
    verify_cleaned_data()

    print("\nDone! Use cleaned dataset at:", OUTPUT_PATH)

RGB DATA CLEANING - FIX ISSUES

Input: /content/RGB
Output: /content/RGB_C

Will remove:
  - Blurry images (score < 100)
  - Low visibility images (< 0.5% edges)
  - Resize all to 224x224

Start cleaning? (yes/no): yes
Creating output directory structure...
Output directory: /content/RGB_C

Processing PATCHES...


  Normal(Healthy skin): 100%|██████████| 543/543 [00:00<00:00, 622.57it/s]



Processing TESTSET...


  TestSet: 100%|██████████| 163/163 [00:01<00:00, 103.06it/s]



Processing TRANSFER-LEARNING...


  Wound Images2: 100%|██████████| 672/672 [00:02<00:00, 290.32it/s]



CLEANING COMPLETE!

Statistics:
  Total images processed: 2169
  Removed (blurry): 568
  Removed (low visibility): 0
  Resized to 224x224: 877
  Final dataset size: 1601 images

  Removal rate: 26.2%

Cleaned data saved to: /content/RGB_C

VERIFYING CLEANED DATA

Total images in cleaned dataset: 767

Resolutions:
  y 224x224: 767 images

ALL IMAGES ARE 224x224!

Done! Use cleaned dataset at: /content/RGB_C


In [ ]:
!zip -r /content/RGB_C.zip /content/RGB_C


  adding: content/RGB_C/ (stored 0%)
  adding: content/RGB_C/Patches/ (stored 0%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/ (stored 0%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/49.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/21.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/48.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/524.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/531.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/120.jpg (deflated 0%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/29.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/329.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/60.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/303.jpg (deflated 1%)
  adding: content/RGB_C/Patches/Normal(Healthy skin)/301.jpg (deflated 1%)
  adding: content/RGB_C/Patches/

In [1]:
!unzip /content/RGB_C.zip

Archive:  /content/RGB_C.zip
   creating: RGB_C/
   creating: RGB_C/Patches/
   creating: RGB_C/Patches/Abnormal(Ulcer)/
  inflating: RGB_C/Patches/Abnormal(Ulcer)/1.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/10.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/100.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/101.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/102.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/103.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/104.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/105.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/106.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/107.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/108.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/109.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/11.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/110.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/111.jpg  
  inflating: RGB_C/Patches/Abnormal(Ulcer)/112.jpg  
  inflating: RGB_C/Patches/Abnormal

In [2]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [3]:
INPUT_DIR  = "/content/RGB_C"
OUTPUT_DIR = "/content/RGB_P"
GAMMA      = 1.2       # >1 brightens, <1 darkens — adjust if needed
CLIP_LIMIT = 2.0       # CLAHE clip limit
TILE_GRID  = (8, 8)    # CLAHE tile grid size
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

In [4]:
# White Balance

def white_balance(img_bgr):
    result = img_bgr.copy().astype(np.float32)
    avg_b  = np.mean(result[:, :, 0])
    avg_g  = np.mean(result[:, :, 1])
    avg_r  = np.mean(result[:, :, 2])
    avg    = (avg_b + avg_g + avg_r) / 3.0
    result[:, :, 0] = np.clip(result[:, :, 0] * (avg / avg_b), 0, 255)
    result[:, :, 1] = np.clip(result[:, :, 1] * (avg / avg_g), 0, 255)
    result[:, :, 2] = np.clip(result[:, :, 2] * (avg / avg_r), 0, 255)
    return result.astype(np.uint8)

In [5]:
# Gamma Correction

def gamma_correction(img_bgr, gamma=1.2):
    inv_gamma = 1.0 / gamma
    table     = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ]).astype(np.uint8)
    return cv2.LUT(img_bgr, table)

In [6]:
# ROI Extraction

def extract_roi(img_bgr):
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Skin tone range in HSV — covers most wound/skin regions
    lower = np.array([0,  20,  70], dtype=np.uint8)
    upper = np.array([25, 255, 255], dtype=np.uint8)
    mask  = cv2.inRange(hsv, lower, upper)

    # Morphological closing to fill gaps in the mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)

    # If mask is mostly empty (bad image), return original
    if np.sum(mask) < 0.05 * mask.size:
        return img_bgr

    result = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    return result


In [7]:
# CLAHE

def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l       = clahe.apply(l)
    lab     = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


In [8]:
# Full Preprocessing Pipeline

def preprocess_image(img_bgr):
    img = white_balance(img_bgr)
    img = gamma_correction(img, gamma=GAMMA)
    img = extract_roi(img)
    img = apply_clahe(img, clip_limit=CLIP_LIMIT, tile_grid=TILE_GRID)
    return img

In [9]:
# final

skipped, processed = 0, 0

for root, dirs, files in os.walk(INPUT_DIR):
    for fname in tqdm(files, desc=f"Processing {os.path.basename(root)}"):
        if Path(fname).suffix.lower() not in VALID_EXTS:
            continue

        src_path = os.path.join(root, fname)

        # Mirror folder structure in output dir
        rel_path = os.path.relpath(src_path, INPUT_DIR)
        dst_path = os.path.join(OUTPUT_DIR, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        img = cv2.imread(src_path)
        if img is None:
            skipped += 1
            continue

        preprocessed = preprocess_image(img)
        cv2.imwrite(dst_path, preprocessed)
        processed += 1

print(f"\nPreprocessing complete.")
print(f"   Processed : {processed} images")
print(f"   Skipped   : {skipped} images (unreadable)")
print(f"   Output    : {OUTPUT_DIR}")

Processing RGB_C: 0it [00:00, ?it/s]
Processing Patches: 0it [00:00, ?it/s]
Processing TestSet: 100%|██████████| 106/106 [00:00<00:00, 172.39it/s]
Processing Transfer-Learning images: 0it [00:00, ?it/s]
Processing Wound Images2: 100%|██████████| 613/613 [00:03<00:00, 191.99it/s]


Preprocessing complete.
   Processed : 1601 images
   Skipped   : 0 images (unreadable)
   Output    : /content/RGB_P


In [10]:
!zip -r /content/RGB_P.zip /content/RGB_P

  adding: content/RGB_P/ (stored 0%)
  adding: content/RGB_P/Patches/ (stored 0%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/ (stored 0%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/436.jpg (deflated 3%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/70.jpg (deflated 3%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/514.jpg (deflated 3%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/406.jpg (deflated 2%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/191.jpg (deflated 3%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/405.jpg (deflated 5%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/356.jpg (deflated 2%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/107.jpg (deflated 5%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/151.jpg (deflated 2%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/452.jpg (deflated 3%)
  adding: content/RGB_P/Patches/Normal(Healthy skin)/305.jpg (deflated 2%)
  adding: content/RGB_P/Patc